In [1]:
!pip install indic-nlp-library polyglot pyicu pycld2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.3/126.3 kB 3.2 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.6/267.6 kB 5.1 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 77.5 MB/s eta 0:00:00:00:0100:01
  Created wheel for polyglot: filename=polyglot-16.7.4-py2.py3-none-any.whl size=52563 sha256=620c62d52ffdcbdbb026fe7dec9b30bd0ea3819082e3936cbbb9f7b284926e54
  Stored in directory: /root/.cache/pip/wheels/70/e3/da/d2f524831513cedfe2a49e46a94028bf6f632f6ba172d6dead
  Created wheel for pyicu: filename=pyicu-2.15.3-cp311-cp311-linux_x86_64.whl size=2715662 sha256=044a8e4334f8753dbae935cce7221eb085785a13f0b9ba847e1652169d25888f
  Stored in directory: /root/.cache/pip/wheels/ad/8d/

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import FeatureUnion
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')

# =========================
# Prerequisites:
# =========================
# pip install pandas scikit-learn tqdm
# Optional for better preprocessing:
# pip install indic-nlp-library polyglot pyicu pycld2
# =========================

try:
    from indicnlp.tokenize import indic_tokenize
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    INDIC_AVAILABLE = True
    print("✅ Indic NLP Library loaded successfully")
except ImportError:
    INDIC_AVAILABLE = False
    print("❌ Indic NLP not available. Install: pip install indic-nlp-library")

try:
    from polyglot.text import Text
    POLYGLOT_AVAILABLE = True
    print("✅ Polyglot loaded successfully")
except ImportError:
    POLYGLOT_AVAILABLE = False
    print("❌ Polyglot not available. Install: pip install polyglot pyicu pycld2")

# =========================
# Professional Bengali Preprocessor
# =========================
class ProfessionalBengaliPreprocessor:
    def __init__(self):
        if INDIC_AVAILABLE:
            self.normalizer = IndicNormalizerFactory().get_normalizer("bn")
        
        self.bengali_stopwords = {
            'আমি', 'তুমি', 'সে', 'তিনি', 'আমরা', 'তোমরা', 'তারা', 'এই', 'এটি', 'ওই', 'ওটি',
            'একটি', 'একটা', 'একজন', 'তার', 'তাদের', 'আমার', 'তোমার', 'আমাদের', 'তাদের',
            'এবং', 'বা', 'অথবা', 'কিন্তু', 'যদি', 'তাহলে', 'না', 'নয়', 'নেই', 'হয়', 'আছে', 
            'ছিল', 'হবে', 'করে', 'করতে', 'করা', 'হতে', 'হওয়া', 'যে', 'যা', 'যার', 'যাদের',
            'কি', 'কী', 'কেন', 'কখন', 'কোথায়', 'কিভাবে', 'কীভাবে', 'সব', 'সকল', 'সমস্ত',
            'অনেক', 'কিছু', 'কোন', 'কোনো', 'দিয়ে', 'থেকে', 'পর্যন্ত', 'জন্য', 'সাথে', 
            'মধ্যে', 'ভিতরে', 'উপর', 'নিচে', 'পাশে', 'কাছে', 'দূরে', 'আগে', 'পরে', 'সময়',
            'এখন', 'তখন', 'এখানে', 'সেখানে', 'ওখানে', 'যেখানে', 'হয়তো', 'অবশ্যই', 'অবশ্য',
            'শুধু', 'শুধুমাত্র', 'প্রায়', 'বেশ', 'বেশি', 'কম', 'খুব', 'অত', 'এত', 'তত', 'যত',
            'ও', 'আর', 'তো', 'যদিও', 'অর্থাৎ', 'ইত্যাদি'
        }
        
        self.advanced_suffixes = [
            ('ছিলাম', ''), ('ছিলেন', ''), ('ছিলো', ''), ('ছিল', ''), ('েছেন', ''), ('েছে', ''), ('েছি', ''), ('েছো', ''),
            ('েবেন', ''), ('েবে', ''), ('েবো', ''), ('েব', ''), ('চ্ছেন', ''), ('চ্ছে', ''), ('চ্ছি', ''), ('চ্ছো', ''),
            ('তেছেন', ''), ('তেছে', ''), ('তেছি', ''), ('তেছো', ''), ('েতেন', ''), ('েত', ''), ('েতে', ''), ('তেন', ''), ('তে', ''),
            ('গুলো', ''), ('গুলি', ''), ('গুলা', ''), ('দের', ''), ('েদের', ''), ('রা', ''), ('েরা', ''), ('গণ', ''),
            ('বৃন্দ', ''), ('মণ্ডলী', ''), ('সমূহ', ''), ('গোষ্ঠী', ''), ('দল', ''),
            ('কে', ''), ('েক', ''), ('র', ''), ('ের', ''), ('েদর', ''), ('তে', ''), ('েত', ''), ('েতে', ''), 
            ('য়', ''), ('েয়', ''), ('নে', ''), ('েন', ''),
            ('িক', ''), ('ীয়', ''), ('িত', ''), ('ীত', ''), ('ীন', ''), ('িন', ''), ('মান', ''), ('বান', ''),
            ('যুক্ত', ''), ('হীন', ''), ('বিহীন', ''), ('রহিত', ''), ('পূর্ণ', ''),
            ('ন্ত', ''), ('িত', ''), ('ীত', ''), ('ানো', ''), ('েনো', ''), ('ুনো', ''),
            ('ত্ব', ''), ('তা', ''), ('ত্বো', ''), ('পনা', ''), ('আমি', ''), ('ানি', ''), ('উনি', ''), ('ী', ''), ('ি', ''), ('া', ''), ('ে', ''), ('ো', '')
        ]
    
    def normalize_text(self, text):
        if pd.isna(text) or text == "":
            return ""
        text = str(text).strip()
        if INDIC_AVAILABLE:
            text = self.normalizer.normalize(text)
        else:
            import unicodedata
            text = unicodedata.normalize('NFKC', text)
        return text
    
    def tokenize_text(self, text):
        if INDIC_AVAILABLE:
            tokens = indic_tokenize.trivial_tokenize(text, lang='bn')
        else:
            tokens = text.split()
        return tokens
    
    def advanced_stem(self, word):
        if len(word) <= 2:
            return word
        original_word = word
        for suffix, replacement in sorted(self.advanced_suffixes, key=lambda x: len(x[0]), reverse=True):
            if word.endswith(suffix) and len(word) > len(suffix) + 2:
                return word[:-len(suffix)] + replacement
        return original_word
    
    def polyglot_stem(self, text):
        if not POLYGLOT_AVAILABLE:
            return text
        try:
            polyglot_text = Text(text, hint_language_code='bn')
            return ' '.join(str(word).lower() for word in polyglot_text.words)
        except:
            return text
    
    def preprocess_text(self, text):
        import re
        text = self.normalize_text(text)
        if not text:
            return ""
        
        text = re.sub(r'[!]{2,}', ' STRONG_EXCLAMATION ', text)
        text = re.sub(r'[!]', ' EXCLAMATION ', text)
        text = re.sub(r'[?]{2,}', ' STRONG_QUESTION ', text)
        text = re.sub(r'[?]', ' QUESTION ', text)
        text = re.sub(r'[.]{3,}', ' ELLIPSIS ', text)
        text = re.sub(r'http\S+|www\S+', ' URL ', text)
        text = re.sub(r'\d+', ' NUMBER ', text)
        text = re.sub(r'[^\u0980-\u09FF\sSTRONG_EXCLAMATIONEXCLAMATIONSTRONG_QUESTIONQUESTIONELLIPSISURLNUMBER]', ' ', text)
        
        tokens = self.tokenize_text(text)
        processed_tokens = []
        for token in tokens:
            token = token.strip()
            if len(token) <= 1 or token.lower() in self.bengali_stopwords:
                continue
            processed_tokens.append(self.advanced_stem(token))
        return ' '.join(processed_tokens).strip()

# =========================
# Feature Union for advanced features
# =========================
def create_advanced_feature_union(**params):
    from sklearn.feature_extraction.text import TfidfVectorizer
    return FeatureUnion([
        ('word_unigrams', TfidfVectorizer(analyzer='word', ngram_range=(1,1), max_features=params.get('word1_features',5000))),
        ('word_bigrams', TfidfVectorizer(analyzer='word', ngram_range=(2,2), max_features=params.get('word2_features',3000))),
        ('char_trigrams', TfidfVectorizer(analyzer='char', ngram_range=(3,3), max_features=params.get('char3_features',4000))),
        ('char_4grams', TfidfVectorizer(analyzer='char', ngram_range=(4,4), max_features=params.get('char4_features',3000))),
        ('char_5grams', TfidfVectorizer(analyzer='char', ngram_range=(5,5), max_features=params.get('char5_features',2000)))
    ])

# =========================
# Load datasets
# =========================
train_df = pd.read_csv('/kaggle/input/datasetvlr/train.csv')
val_df   = pd.read_csv('/kaggle/input/datasetvlr/validation.csv')
test_df  = pd.read_csv('/kaggle/input/datasetvlr/test.csv')

preprocessor = ProfessionalBengaliPreprocessor()

# Preprocess sentences
X_train = train_df['clean_sentence'].apply(preprocessor.preprocess_text)
y_train = train_df['intensity_level_normalized']

X_val = val_df['clean_sentence'].apply(preprocessor.preprocess_text)
y_val = val_df['intensity_level_normalized']

X_test = test_df['clean_sentence'].apply(preprocessor.preprocess_text)
y_test = test_df['intensity_level_normalized']

# =========================
# Vectorization
# =========================
vectorizer = create_advanced_feature_union(
    word1_features=10000, word2_features=8000, char3_features=8000, char4_features=6000, char5_features=4000
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

# =========================
# Train Logistic Regression
# =========================
model = LogisticRegression(
    C=5.0, penalty='l2', max_iter=8000, solver='newton-cg', random_state=42, multi_class='ovr'
)

model.fit(X_train_vec, y_train)

# =========================
# Validation
# =========================
val_preds = model.predict(X_val_vec)
print("📊 Validation Accuracy:", accuracy_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

# =========================
# Test
# =========================
test_preds = model.predict(X_test_vec)
print("📊 Test Accuracy:", accuracy_score(y_test, test_preds))
print(classification_report(y_test, test_preds))


✅ Indic NLP Library loaded successfully
✅ Polyglot loaded successfully
📊 Validation Accuracy: 0.7009925558312655
              precision    recall  f1-score   support

           0       0.62      0.56      0.59       273
           1       0.54      0.67      0.60       232
           2       0.63      0.72      0.67       261
           3       0.77      0.76      0.77       274
           4       0.81      0.75      0.78       280
           5       0.86      0.73      0.79       292

    accuracy                           0.70      1612
   macro avg       0.71      0.70      0.70      1612
weighted avg       0.71      0.70      0.70      1612

📊 Test Accuracy: 0.6809169764560099
              precision    recall  f1-score   support

           0       0.58      0.61      0.59       274
           1       0.55      0.64      0.59       233
           2       0.63      0.61      0.62       261
           3       0.73      0.72      0.73       274
           4       0.75      0.72    